In [23]:
import math
import pickle
import time
import osmnx as ox
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

UMBRAL_TRAMO_CORTADO = 1e11


with open("graph_aws.pkl", "rb") as f:
    G = pickle.load(f)

print(f"Grafo: {G.number_of_nodes()} vértices, {G.number_of_edges()} aristas")

sin_peso = sum(1 for _, _, d in G.edges(data=True) if d.get("weight") is None)
assert sin_peso == 0, f"{sin_peso} aristas sin peso; usa el grafo operativo"


FACTOR_MINIMO = min(
    d["weight"] / d["length"]
    for _, _, d in G.edges(data=True)
    if d.get("length", 0) > 0 and d["weight"] < UMBRAL_TRAMO_CORTADO
)
print(f"Factor mínimo de la función de coste: {FACTOR_MINIMO:.4f}")


def haversine_m(lat1, lon1, lat2, lon2):
    r = 6371000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def heuristica(graph, destino, factor=1.0, activa=True, registro=None):
    lat_d = graph.nodes[destino]["y"]
    lon_d = graph.nodes[destino]["x"]

    def h(n, _):
        if registro is not None:
            registro.add(n)
        if not activa:
            return 0.0
        return factor * haversine_m(
            graph.nodes[n]["y"], graph.nodes[n]["x"], lat_d, lon_d
        )

    return h


def resolver(graph, origen, destino, atributo, factor, con_heuristica):
    """Resuelve una ruta y devuelve trayecto, tiempo y vértices explorados."""
    visitados = set()
    h = heuristica(graph, destino, factor,
                   activa=con_heuristica, registro=visitados)

    t0 = time.perf_counter()
    ruta = nx.astar_path(graph, origen, destino, heuristic=h, weight=atributo)
    ms = (time.perf_counter() - t0) * 1000

    return ruta, ms, visitados



PARES = [
    ("Estación del Norte → Ciudad de las Artes",(39.4658, -0.3776), (39.4540, -0.3530)),
    ("Torres de Serranos → Puerto", (39.4790, -0.3757), (39.4600, -0.3300)),
    ("Malvarrosa → Nuevo Centro", (39.4720, -0.3250), (39.4750, -0.3900)),
    ("Benimaclet → Patraix", (39.4870, -0.3620), (39.4590, -0.4010)),
    ("Campanar → Cabanyal", (39.4820, -0.4020), (39.4680, -0.3280)),
    ("Ayuntamiento → Mestalla", (39.4692, -0.3768), (39.4719, -0.3594)),
    ("Ruzafa → Universitat Politècnica", (39.4600, -0.3720), (39.4810, -0.3390)),
]

REPETICIONES = 5


# Ejecucion
filas = []
mapas = {} # guarda los vértices explorados del primer par, para la figura

for idx, (nombre, (olat, olon), (dlat, dlon)) in enumerate(PARES):

    o = ox.distance.nearest_nodes(G, X=olon, Y=olat)
    d = ox.distance.nearest_nodes(G, X=dlon, Y=dlat)

    if o == d:
        print(f"[aviso] «{nombre}»: origen y destino coinciden, se omite")
        continue

    registro = {}

    for etiqueta, atributo, factor in [
        ("distancia", "length", 1.0),
        ("coste", "weight", FACTOR_MINIMO),
    ]:
        for algoritmo, con_h in [("dijkstra", False), ("astar", True)]:

            tiempos = []
            for _ in range(REPETICIONES):
                ruta, ms, vis = resolver(G, o, d, atributo, factor, con_h)
                tiempos.append(ms)

            registro[(etiqueta, algoritmo)] = {
                "ruta": ruta,
                "ms": float(np.median(tiempos)),
                "explorados": len(vis),
                "vertices": vis,
            }

    if idx == 0:
        mapas = registro

    for etiqueta in ("distancia", "coste"):
        dij = registro[(etiqueta, "dijkstra")]
        ast = registro[(etiqueta, "astar")]

        filas.append({
            "par": nombre,
            "criterio": etiqueta,
            "t_dijkstra": dij["ms"],
            "t_astar": ast["ms"],
            "reduccion_t": 100 * (dij["ms"] - ast["ms"]) / dij["ms"],
            "n_dijkstra": dij["explorados"],
            "n_astar": ast["explorados"],
            "reduccion_n": 100 * (dij["explorados"] - ast["explorados"]) / dij["explorados"],
            "identicas": dij["ruta"] == ast["ruta"],
            "saltos": len(dij["ruta"]),
        })

import pandas as pd
res = pd.DataFrame(filas)


print("\n" + "=" * 78)
print("RESULTADOS POR PAR")
print("=" * 78)
print(res.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

print("\n" + "=" * 78)
print("AGREGADO POR CRITERIO")
print("=" * 78)
agg = res.groupby("criterio").agg(
    t_dijkstra=("t_dijkstra", "mean"),
    t_astar=("t_astar", "mean"),
    reduccion_t=("reduccion_t", "mean"),
    n_dijkstra=("n_dijkstra", "mean"),
    n_astar=("n_astar", "mean"),
    reduccion_n=("reduccion_n", "mean"),
)
print(agg.to_string(float_format=lambda v: f"{v:.2f}"))

coinciden = res.identicas.all()
print(f"\n¿Todas las rutas coinciden entre ambos algoritmos? {coinciden}")
if not coinciden:
    print("ATENCIÓN: hay discrepancias. Revisa la admisibilidad de la heurística.")
    print(res[~res.identicas][["par", "criterio"]].to_string(index=False))


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))

x = np.arange(len(agg))
ancho = 0.36

ax1.bar(x - ancho / 2, agg.t_dijkstra, ancho, label="Dijkstra",
        color="#95a5a6", edgecolor="white")
ax1.bar(x + ancho / 2, agg.t_astar, ancho, label="A*",
        color="#c0563b", edgecolor="white")
ax1.set_xticks(x)
ax1.set_xticklabels(["Ruta más corta", "Ruta óptima"])
ax1.set_ylabel("Tiempo de ejecucción (ms)")
ax1.set_title("(a) Tiempo medio de ejecucción")
ax1.legend(frameon=False)
ax1.grid(axis="y", alpha=0.3)
ax1.set_axisbelow(True)

for i, (dij, ast) in enumerate(zip(agg.t_dijkstra, agg.t_astar)):
    ax1.annotate(f"$-${100*(dij-ast)/dij:.0f}%",
                 (i + ancho / 2, ast), xytext=(0, 5),
                 textcoords="offset points", ha="center",
                 fontsize=9, color="#c0563b", weight="bold")

ax2.bar(x - ancho / 2, agg.n_dijkstra, ancho, label="Dijkstra",
        color="#95a5a6", edgecolor="white")
ax2.bar(x + ancho / 2, agg.n_astar, ancho, label="A*",
        color="#c0563b", edgecolor="white")
ax2.set_xticks(x)
ax2.set_xticklabels(["Ruta más corta", "Ruta óptima"])
ax2.set_ylabel("Vértices explorados")
ax2.set_title("(b) Vértices considerados por la búsqueda")
ax2.legend(frameon=False)
ax2.grid(axis="y", alpha=0.3)
ax2.set_axisbelow(True)

for i, (dij, ast) in enumerate(zip(agg.n_dijkstra, agg.n_astar)):
    ax2.annotate(f"$-${100*(dij-ast)/dij:.0f}%",
                 (i + ancho / 2, ast), xytext=(0, 5),
                 textcoords="offset points", ha="center",
                 fontsize=9, color="#c0563b", weight="bold")

for ax in (ax1, ax2):
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)

plt.tight_layout()
plt.savefig("comparativa_astar_dijkstra.png", dpi=200, bbox_inches="tight")
plt.close()


if mapas:
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    dij = mapas[("coste", "dijkstra")]
    ast = mapas[("coste", "astar")]

    for ax, datos, titulo in [
        (axes[0], dij, f"(a) Dijkstra — {len(dij['vertices'])} vértices"),
        (axes[1], ast, f"(b) A* — {len(ast['vertices'])} vértices"),
    ]:
        # Red viaria de fondo
        xs = [G.nodes[n]["x"] for n in G.nodes]
        ys = [G.nodes[n]["y"] for n in G.nodes]
        ax.scatter(xs, ys, s=0.4, color="#dddddd", zorder=1)

        # Vértices explorados
        vx = [G.nodes[n]["x"] for n in datos["vertices"]]
        vy = [G.nodes[n]["y"] for n in datos["vertices"]]
        ax.scatter(vx, vy, s=2.5, color="#5b8fa8", alpha=0.65, zorder=2)

        # Ruta resultante
        rx = [G.nodes[n]["x"] for n in datos["ruta"]]
        ry = [G.nodes[n]["y"] for n in datos["ruta"]]
        ax.plot(rx, ry, color="#c0563b", linewidth=2, zorder=3)

        # Extremos
        ax.scatter([rx[0]], [ry[0]], s=30, color="#27ae60",
                   edgecolor="white", zorder=4)
        ax.scatter([rx[-1]], [ry[-1]], s=30, color="#8e44ad",
                   edgecolor="white", zorder=4)

        ax.set_title(titulo, fontsize=11)
        ax.set_aspect("equal")
        ax.set_axis_off()

    plt.tight_layout()
    plt.savefig("mapa_exploracion.png", dpi=200, bbox_inches="tight")
    plt.close()




Grafo: 8091 vértices, 14571 aristas
Factor mínimo de la función de coste: 0.1446

RESULTADOS POR PAR
                                     par  criterio  t_dijkstra  t_astar  reduccion_t  n_dijkstra  n_astar  reduccion_n  identicas  saltos
Estación del Norte → Ciudad de las Artes distancia       14.94     0.85        94.34        3849      263        93.17       True      54
Estación del Norte → Ciudad de las Artes     coste       14.96     1.85        87.64        4002      295        92.63       True      53
             Torres de Serranos → Puerto distancia       20.73     1.55        92.50        5125      401        92.18       True      75
             Torres de Serranos → Puerto     coste       18.50     2.23        87.96        5093      401        92.13       True      75
               Malvarrosa → Nuevo Centro distancia       18.01     8.01        55.51        4538     1552        65.80       True     102
               Malvarrosa → Nuevo Centro     coste       16.44     8.28

In [22]:
print("\n" + "=" * 78)
print("FILAS PARA LA TABLA DE LA MEMORIA")
print("=" * 78)
for _, r in res.iterrows():
    print(f"    {r.par} & {r.criterio} & {r.t_dijkstra:.1f} & "
          f"{r.t_astar:.1f} & {r.reduccion_t:.0f}\\% & "
          f"{r.n_dijkstra} & {r.n_astar} & "
          f"{'Sí' if r.identicas else 'No'} \\\\")
    print("    \\hline")


FILAS PARA LA TABLA DE LA MEMORIA
    Estación del Norte → Ciudad de las Artes & distancia & 15.6 & 1.0 & 94\% & 3849 & 263 & Sí \\
    \hline
    Estación del Norte → Ciudad de las Artes & coste & 14.8 & 1.1 & 93\% & 4002 & 295 & Sí \\
    \hline
    Torres de Serranos → Puerto & distancia & 21.6 & 2.4 & 89\% & 5125 & 401 & Sí \\
    \hline
    Torres de Serranos → Puerto & coste & 19.3 & 1.6 & 92\% & 5093 & 401 & Sí \\
    \hline
    Malvarrosa → Nuevo Centro & distancia & 18.9 & 11.4 & 40\% & 4538 & 1552 & Sí \\
    \hline
    Malvarrosa → Nuevo Centro & coste & 18.3 & 7.8 & 57\% & 4520 & 1552 & Sí \\
    \hline
    Benimaclet → Patraix & distancia & 24.3 & 5.7 & 77\% & 5536 & 955 & Sí \\
    \hline
    Benimaclet → Patraix & coste & 21.9 & 4.8 & 78\% & 5508 & 951 & Sí \\
    \hline
    Campanar → Cabanyal & distancia & 30.2 & 18.9 & 37\% & 6771 & 3104 & Sí \\
    \hline
    Campanar → Cabanyal & coste & 37.0 & 19.6 & 47\% & 6707 & 2806 & Sí \\
    \hline
    Ayuntamiento → Mestall